In [1]:
# =============================================================================
# Project: Multimodal Sequential Recommendation
# Notebook: 03_Data_Preprocessing.ipynb
# Author: Manit Chitnukul
# Description: 
#   Performs rigorous data cleaning:
#   1. Filter items without images.
#   2. Apply iterative k-core filtering (k=5) to ensure dense data.
#   3. Save the clean dataset for model training.
# =============================================================================

import os
import gzip
import json
import pandas as pd
from tqdm import tqdm

# --- 1. CONFIGURATION ---
BASE_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, 'data')

# Input Files
REVIEW_FILE = os.path.join(DATA_DIR, 'Luxury_Beauty.json.gz')
META_FILE = os.path.join(DATA_DIR, 'meta_Luxury_Beauty.json.gz')

# Output File (Clean Data)
OUTPUT_FILE = os.path.join(DATA_DIR, 'clean_Luxury_Beauty_5core.csv')

def preprocess_data():
    print("🚀 Starting Data Preprocessing Pipeline...")
    
    # --- Step 1: Extract Valid Items (Must have Image) ---
    print("\n[Step 1] Extracting items with images from Metadata...")
    valid_items = {} # Store {asin: image_url}
    
    if not os.path.exists(META_FILE):
        print(f"❌ Error: Meta file not found: {META_FILE}")
        return

    with gzip.open(META_FILE, 'rt', encoding='utf-8') as f:
        for line in tqdm(f, desc="Scanning Metadata"):
            try:
                data = json.loads(line.strip())
                asin = data.get('asin')
                
                # Check for Image
                imgs = data.get('imageURL', []) or data.get('image', [])
                
                if asin and isinstance(imgs, list) and len(imgs) > 0:
                    # Keep the first image
                    valid_items[asin] = imgs[0]
            except json.JSONDecodeError:
                continue
    
    print(f"   ✅ Found {len(valid_items):,} items with valid images.")

    # --- Step 2: Load & Filter Reviews (Remove No-Image Items) ---
    print("\n[Step 2] Loading Reviews and filtering no-image items...")
    
    interactions = []
    
    if not os.path.exists(REVIEW_FILE):
        print(f"❌ Error: Review file not found: {REVIEW_FILE}")
        return

    with gzip.open(REVIEW_FILE, 'rt', encoding='utf-8') as f:
        for line in tqdm(f, desc="Loading Reviews"):
            try:
                data = json.loads(line.strip())
                
                user = data.get('reviewerID')
                item = data.get('asin')
                time = data.get('unixReviewTime')
                rating = data.get('overall')
                
                # Filter: Keep only if item has an image
                if user and item and item in valid_items:
                    interactions.append({
                        'user_id': user,
                        'item_id': item,
                        'rating': rating,
                        'timestamp': time,
                        'image_url': valid_items[item]
                    })
            except json.JSONDecodeError:
                continue
                
    # Convert to DataFrame for easier processing
    df = pd.DataFrame(interactions)
    print(f"   ✅ Initial DataFrame shape: {df.shape}")
    print(f"      (Users: {df['user_id'].nunique():,}, Items: {df['item_id'].nunique():,})")

    # --- Step 3: Iterative 5-core Filtering ---
    print("\n[Step 3] Applying Iterative 5-core Filtering...")
    # Logic: Remove users < 5 -> items might drop < 5 -> remove items -> users might drop < 5... repeat.
    
    k = 5
    iteration = 0
    while True:
        iteration += 1
        print(f"   🔄 Iteration {iteration}:")
        
        # 1. Filter Users
        user_counts = df['user_id'].value_counts()
        valid_users = user_counts[user_counts >= k].index
        df_filtered = df[df['user_id'].isin(valid_users)].copy()
        
        n_users_dropped = len(df) - len(df_filtered)
        print(f"      - Dropped {n_users_dropped:,} interactions from inactive users.")
        
        # 2. Filter Items
        item_counts = df_filtered['item_id'].value_counts()
        valid_items_core = item_counts[item_counts >= k].index
        df_filtered = df_filtered[df_filtered['item_id'].isin(valid_items_core)].copy()
        
        n_items_dropped = len(df) - len(df_filtered) - n_users_dropped
        print(f"      - Dropped {n_items_dropped:,} interactions from unpopular items.")
        
        # Check convergence
        if len(df) == len(df_filtered):
            print("   ✅ Converged! No more users/items to drop.")
            break
        
        df = df_filtered
        print(f"      -> Remaining: {len(df):,} interactions.")

    # --- Step 4: Final Statistics & Save ---
    print("\n[Step 4] Final Dataset Statistics & Saving...")
    n_users = df['user_id'].nunique()
    n_items = df['item_id'].nunique()
    n_interactions = len(df)
    sparsity = (1 - (n_interactions / (n_users * n_items))) * 100
    
    print("="*40)
    print("📉 CLEAN DATASET REPORT")
    print("="*40)
    print(f"👥 Users:        {n_users:,}")
    print(f"📦 Items:        {n_items:,}")
    print(f"🔁 Interactions: {n_interactions:,}")
    print(f"🌌 Sparsity:     {sparsity:.4f}%")
    print("="*40)
    
    # Save to CSV
    print(f"💾 Saving to {OUTPUT_FILE}...")
    df.to_csv(OUTPUT_FILE, index=False)
    print("✅ Done! ready for Model Training.")

if __name__ == "__main__":
    preprocess_data()

🚀 Starting Data Preprocessing Pipeline...

[Step 1] Extracting items with images from Metadata...


Scanning Metadata: 12299it [00:00, 80397.09it/s]


   ✅ Found 10,075 items with valid images.

[Step 2] Loading Reviews and filtering no-image items...


Loading Reviews: 574628it [00:02, 270754.78it/s]


   ✅ Initial DataFrame shape: (534602, 5)
      (Users: 391,059, Items: 10,075)

[Step 3] Applying Iterative 5-core Filtering...
   🔄 Iteration 1:
      - Dropped 490,920 interactions from inactive users.
      - Dropped 6,137 interactions from unpopular items.
      -> Remaining: 37,545 interactions.
   🔄 Iteration 2:
      - Dropped 4,804 interactions from inactive users.
      - Dropped 1,030 interactions from unpopular items.
      -> Remaining: 31,711 interactions.
   🔄 Iteration 3:
      - Dropped 1,295 interactions from inactive users.
      - Dropped 399 interactions from unpopular items.
      -> Remaining: 30,017 interactions.
   🔄 Iteration 4:
      - Dropped 472 interactions from inactive users.
      - Dropped 154 interactions from unpopular items.
      -> Remaining: 29,391 interactions.
   🔄 Iteration 5:
      - Dropped 200 interactions from inactive users.
      - Dropped 51 interactions from unpopular items.
      -> Remaining: 29,140 interactions.
   🔄 Iteration 6:
  